## Graph Drawing

![](img/Strichfiguren.jpeg)

Introduction

---

### Load dataset

In [ ]:
# Importing Libraries and Loading Data
import pandas as pd
import numpy as np
#from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import matplotlib.pyplot as plt
import scipy
#import seaborn as sns
from collections import Counter
import warnings
import networkx as nx
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

Load dataset.

In [ ]:
# Load dataset
df = pd.read_csv('./datasets/Netflix Streaming Data/Netflix Streaming Data.csv')
print(df.info())
df.head()

Consider only entries with non-null "cast" entries

In [ ]:
df_cast = df.dropna(subset=['cast'])
df_cast = df_cast.reset_index(drop=True)
df_cast.info()

Consider only entries of type = "Movie".

In [ ]:
df_movie = df_cast[df_cast['type'] == 'Movie']
df_movie = df_movie.reset_index(drop=True)
df_movie.info()

Prepare dataframe for constructing the bipartite graph: Remove all columns except "title" and "cast".

In [ ]:
df_bipartite = df_movie[['title', 'cast']]
df_bipartite = df_bipartite.reset_index(drop=True)
df_bipartite.info()

Summary

In [ ]:
titles = set(df_bipartite['title'].unique())
actors = set()
for idx, row in df_bipartite.iterrows():
    actors.update([actor.strip() for actor in row['cast'].split(',')])
print(f"Total unique titles: {len(titles)}")
print(f"Total unique actors: {len(actors)}")

Make list of sample list of actors.

In [ ]:
actors = df_bipartite[df_bipartite['title'] == 'Sankofa']['cast'].values[0].split(',')
actors = [actor.strip() for actor in actors]
actors

---

### Create bipartite graph B.

In [ ]:
B = nx.Graph()

for idx, row in df_bipartite.iterrows():
    title = row['title']
    actors = [actor.strip() for actor in row['cast'].split(',')]
    for actor in actors:
        B.add_edge(title, actor)
for node in list(B.nodes(data=True))[200:250]:
    print(node)

#### Check if graph is bipartite

Key concept: A graph is bipartite if and only if it can be colored with 2 colors such that no adjacent nodes have the same color (equivalently, it contains no odd-length cycles).

For the movie-actor graph, B should definitely be bipartite since it connects titles to actors with no actor-to-actor or title-to-title edges.

To check if a graph is bipartite check for odd cycles.

(Bipartite graphs do not contain odd cycles because all edges in such graphs strictly connect distinct, disjoint vertex sets `A` and `B`. Any path starting in `A`, passing through an odd number of edges (e.g., `A` → `B` → `A` → `B`)), must end in a different partition `B`, making it impossible to return to the starting vertex in an odd number of steps.) Example: In the graph below, count the number of edges you pass, if you leave node `1` and return to that node.

![](img/cycles.jpg)

In [ ]:
# If the graph has odd cycles, this will be slow or fail
cycle_basis = nx.cycle_basis(B)
has_odd_cycle = any(len(cycle) % 2 == 1 for cycle in cycle_basis)
if has_odd_cycle:
    print("Graph is NOT bipartite (contains odd cycles)")
else:
    print("Graph is bipartite")

If not: Check for overlap in "title" and "actor" sets.

Overlaps occur if a movie title is also an actor name. 
I silently removed this problem in the original dataset.

In [ ]:
# Nodes that should be titles
titles = set(df_bipartite['title'].unique())

# Nodes that should be actors
actors = set()
for idx, row in df_bipartite.iterrows():
    actors.update([actor.strip() for actor in row['cast'].split(',')])

# Find overlaps
overlap = titles & actors
if overlap:
    print(f"⚠ Found {len(overlap)} nodes in both sets:")
    for node in overlap:
        print(f"  '{node}'")

Next step: Find the titles of those movies that do not share actors with any other movie.

If a movie appears in a connected component with only 1 title node, all of its actors are unique to that movie and don't appear in any other movie in the dataset.

In [ ]:
titles_set = set(df_bipartite['title'].unique())
components_B = list(nx.connected_components(B))

no_of_single_title_components = 0
no_of_actors_in_single_title_components = 0

for component in components_B:
    titles_in_comp = [node for node in component if node in titles_set]
    
    if len(titles_in_comp) == 1:
        no_of_single_title_components += 1
        movie = titles_in_comp[0]
        # Get the cast from the original dataframe
        cast = df_bipartite[df_bipartite['title'] == movie]['cast'].values[0]
        no_of_actors_in_single_title_components += len(cast.split(',')) 
        #print(f"{movie}")
        #print(f"  Cast: {cast}\n")

print(f"Total number of single-title components: {no_of_single_title_components}")
print(f"Total number of actors in single-title components: {no_of_actors_in_single_title_components}\n")

---

### Create projected graph A.

In [ ]:
from networkx.algorithms import bipartite

# Get all actors from the bipartite graph
actors = set()
for idx, row in df_bipartite.iterrows():
    actors.update([actor.strip() for actor in row['cast'].split(',')])

# Project onto actors
A = bipartite.weighted_projected_graph(B, actors)
list(A.edges(data=True))[:10]  # Display first 10 edges with weights

Count connected components in graph A.

In [ ]:
components = list(nx.connected_components(A))
print(f"Number of connected components: {len(components)}\n")

sum_nodes = sum(len(component) for component in components)
print(f"Total number of actor nodes across all connected components: {sum_nodes}")

#for i, component in enumerate(components, 1):
#    print(f"Connected component {i}: {len(component)} nodes")
#    if len(component) <= 5:
#        print(f"  Nodes: {component}")

Find connected components with 5 or less actor nodes.

In [ ]:
titles_set = set(df_bipartite['title'].unique())
components_B = list(nx.connected_components(B))

small_components = [c for c in components_B if len(c) <= 5]
print(f"Found {len(small_components)} components with 5 or less nodes\n")

#for i, component in enumerate(small_components, 1):
#    titles = [n for n in component if n in titles_set]
#    actors = [n for n in component if n not in titles_set]
#    print(f"Component {i} ({len(component)} nodes):")
#    print(f"  Movies: {titles}")
#    print(f"  Actors: {actors}\n")

Create initial visualization and save to file.

In [ ]:
plt.figure(figsize=(200, 200))
pos = nx.spring_layout(A, k=2, iterations=10)
nx.draw(A, pos, with_labels=True, node_color='lightcoral', node_size=300, font_size=6, width=0.5)
plt.savefig('graphs/actor_graph.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

---

### Analyze graph A further.

In [ ]:
degrees = dict(A.degree())
degree_df = pd.DataFrame(list(degrees.items()), columns=['actor', 'degree'])
degree_df = degree_df.sort_values(by='degree', ascending=False)
top_24 = degree_df.head(24)
print (top_24)

# Create and save
plt.figure(figsize=(14, 8))
plt.barh(top_24['actor'], top_24['degree'], color='orange')
plt.xlabel('Degree')
plt.ylabel('Actor')
plt.title('Top 24 Actors by Co-actor Connections')
plt.gca().invert_yaxis()
plt.savefig('graphs/degree_distribution_top24.svg', format='svg', bbox_inches='tight')
plt.show()

![](img/Anupam%20Kher.webp)

Anupam Kher

![](img/Shah_Rukh_Khan.jpg)

Shah Rukh Khan

End of line

In [ ]:
degrees = dict(A.degree())
degree_df = pd.DataFrame(list(degrees.items()), columns=['actor', 'degree'])
degree_df = degree_df.sort_values(by='degree', ascending=False)
bottom_24 = degree_df.tail(24)
print (bottom_24)

# Create and save
plt.figure(figsize=(14, 8))
plt.barh(bottom_24['actor'], bottom_24['degree'], color='orange')
plt.xlabel('Degree')
plt.ylabel('Actor')
plt.title('Bottom 24 Actors by Co-actor Connections')
plt.gca().invert_yaxis()
plt.savefig('graphs/degree_distribution_bottom24.svg', format='svg', bbox_inches='tight')
plt.show()

Who wants to have Ronnie Coleman as co-actor?

![](img/Ronnie_Coleman.png)

---

### Graphviz

![](img/NetworkX_to_svg.png)

We are now generating a graph visualization with Graphviz. This gives us more freedom in graph styling.